# Visualizing Neural Networks

This notebook explores various techniques to visualize neural networks, from basic architecture representations to advanced interpretability methods. Understanding how to visualize neural networks is crucial for debugging, optimizing, and communicating model behavior.

## Objectives
- Learn different ways to visualize neural network architectures
- Understand techniques for visualizing weights and activations
- Explore advanced interpretability methods
- Create interactive visualizations for neural networks

## 1. Import Required Libraries

We need various libraries for neural network creation and visualization:
- TensorFlow/Keras for building neural networks
- Matplotlib and Plotly for general visualization
- NumPy for numerical operations
- Scikit-learn for datasets and evaluation metrics
- Specialized visualization tools for neural networks

In [ ]:
# Core libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Deep learning libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, Input, Reshape
from tensorflow.keras.utils import plot_model

# Scikit-learn for datasets and metrics
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits, fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

# Visualization tools
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.gridspec as gridspec

# For visualizing decision boundaries
from mlxtend.plotting import plot_decision_regions

# Check if we're running in a Jupyter environment
try:
    from IPython.display import display, HTML, Image
    in_notebook = True
except ImportError:
    in_notebook = False

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tf.keras.__version__}")

## 2. Neural Network Visualization Fundamentals

Neural networks are often described as "black boxes" because their internal workings can be difficult to understand. Visualization techniques help make neural networks more interpretable by:

1. **Architecture Visualization**: Understanding the structure and connections between layers
2. **Weight Visualization**: Examining learned parameters
3. **Activation Visualization**: Seeing how neurons respond to inputs
4. **Feature Visualization**: Understanding what features the network detects
5. **Decision Boundary Visualization**: Visualizing how the model makes decisions
6. **Interpretability Methods**: Using techniques to explain predictions

Visualization is essential for:
- Debugging network behavior
- Model optimization
- Identifying potential biases
- Communicating model behavior to non-technical stakeholders
- Understanding what the model has learned

## 3. Creating a Simple Neural Network

Before we can visualize a neural network, we need to create one. We'll build a simple feed-forward neural network using TensorFlow/Keras on the MNIST dataset of handwritten digits. This model will serve as the basis for our visualization techniques.

In [ ]:
# Load MNIST dataset
# Option 1: Using TensorFlow/Keras
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Preprocess the data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape for the neural network
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

# Checking shapes
print(f"x_train shape: {x_train.shape}")
print(f"x_train_flat shape: {x_train_flat.shape}")
print(f"y_train shape: {y_train.shape}")

# Display a few sample images
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Build a simple fully-connected neural network for MNIST classification
simple_model = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(128, activation='relu', name='hidden1'),
    keras.layers.Dense(64, activation='relu', name='hidden2'),
    keras.layers.Dense(10, activation='softmax', name='output')
])

# Build a CNN model for comparison
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', name='conv1'),
    keras.layers.MaxPooling2D(pool_size=(2, 2), name='pool1'),
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu', name='conv2'),
    keras.layers.MaxPooling2D(pool_size=(2, 2), name='pool2'),
    keras.layers.Flatten(name='flatten'),
    keras.layers.Dense(128, activation='relu', name='dense1'),
    keras.layers.Dense(10, activation='softmax', name='output')
])

# Compile the models
simple_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summaries
print("Simple Neural Network Architecture:")
simple_model.summary()
print("\nCNN Architecture:")
cnn_model.summary()

# We won't train the models fully to save time, but we'll initialize them
# For an actual analysis, you would want to train the models first
# simple_model.fit(x_train_flat, y_train, epochs=5, batch_size=128, validation_split=0.1)
# x_train_cnn = x_train.reshape(x_train.shape[0], 28, 28, 1)
# cnn_model.fit(x_train_cnn, y_train, epochs=5, batch_size=128, validation_split=0.1)

## 4. Visualizing Network Architecture

There are several ways to visualize the architecture of a neural network:
1. **Text-based summary**: Simple but not visual (we've already seen this)
2. **Keras `plot_model`**: Creates a graphical representation
3. **TensorBoard**: Interactive visualization
4. **Custom visualizations**: More control over the presentation

Let's start with some basic approaches:

In [ ]:
# Attempt to visualize the model architecture using Keras plot_model
# This requires graphviz or pydot to be installed

try:
    # Create a visualization of the simple model
    plot_model(simple_model, to_file='simple_model.png', show_shapes=True, show_layer_names=True)
    
    # Display the image
    if in_notebook:
        display(Image('simple_model.png'))
    
    # Create a visualization of the CNN model
    plot_model(cnn_model, to_file='cnn_model.png', show_shapes=True, show_layer_names=True)
    
    # Display the image
    if in_notebook:
        display(Image('cnn_model.png'))
except Exception as e:
    print(f"Error generating model plot: {e}")
    print("If you want to visualize the model architecture using plot_model, you need to install graphviz and pydot:")
    print("pip install pydot graphviz")
    print("And on Linux/macOS, also run: sudo apt-get install graphviz")

In [ ]:
# Create a custom visualization of the simple neural network
def visualize_simple_nn(model):
    # Get layer information
    input_layer = model.layers[0]
    hidden_layers = model.layers[1:-1]
    output_layer = model.layers[-1]
    
    # Create figure
    fig = plt.figure(figsize=(12, 8))
    
    # Calculate positions for nodes
    layers_sizes = [input_layer.input_shape[1]] + [layer.units for layer in model.layers]
    
    # Define layer positions
    layer_positions = np.linspace(0.1, 0.9, len(layers_sizes))
    
    # Maximum number of neurons to display per layer
    max_neurons = 10
    
    # Colors for different layers
    colors = ['#FFC0CB', '#ADD8E6', '#90EE90', '#FFFFE0']
    
    # Draw connections first (so they're behind the nodes)
    for l in range(len(layers_sizes)-1):
        # Draw lines between neurons in adjacent layers
        # For simplicity, we'll only draw a few representative connections
        input_neurons = min(layers_sizes[l], max_neurons)
        output_neurons = min(layers_sizes[l+1], max_neurons)
        
        for i in range(input_neurons):
            # Calculate y position for neuron in the current layer
            y1 = 0.9 - 0.8 * (i / (input_neurons - 1)) if input_neurons > 1 else 0.5
            
            for j in range(output_neurons):
                # Calculate y position for neuron in the next layer
                y2 = 0.9 - 0.8 * (j / (output_neurons - 1)) if output_neurons > 1 else 0.5
                
                # Draw a line between them
                plt.plot([layer_positions[l], layer_positions[l+1]], [y1, y2], 'gray', alpha=0.2)
    
    # Draw neurons for each layer
    for l, layer_size in enumerate(layers_sizes):
        # Determine how many neurons to show
        display_neurons = min(layer_size, max_neurons)
        
        # Calculate positions
        x = layer_positions[l]
        for i in range(display_neurons):
            y = 0.9 - 0.8 * (i / (display_neurons - 1)) if display_neurons > 1 else 0.5
            circle = plt.Circle((x, y), 0.02, color=colors[l % len(colors)], zorder=4)
            fig.add_artist(circle)
        
        # If we're not showing all neurons, indicate that with an ellipsis
        if layer_size > max_neurons:
            plt.text(x, 0.1, f"...{layer_size - max_neurons} more", ha='center', va='center')
    
    # Add layer labels
    plt.text(layer_positions[0], 0.95, f'Input Layer\n({layers_sizes[0]} neurons)', ha='center')
    
    for i, layer in enumerate(hidden_layers):
        plt.text(layer_positions[i+1], 0.95, f'Hidden Layer {i+1}\n({layers_sizes[i+1]} neurons)', ha='center')
    
    plt.text(layer_positions[-1], 0.95, f'Output Layer\n({layers_sizes[-1]} neurons)', ha='center')
    
    # Remove axes
    plt.axis('off')
    plt.title('Neural Network Architecture', fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize the simple model
visualize_simple_nn(simple_model)

## 5. Visualizing Weights and Activations

Understanding the weights and activations of a neural network can provide insights into what the network has learned. Let's visualize:
1. **Weight distributions**: Histograms of weights between layers
2. **Weight matrices**: Visualizing connection strengths
3. **Activations**: How neurons respond to specific inputs

First, let's initialize our model with random weights so we can visualize them:

In [ ]:
# Get the weights from each layer
weights_list = [layer.get_weights() for layer in simple_model.layers if layer.get_weights()]

# Create a figure for displaying weight distributions
plt.figure(figsize=(15, 10))

for i, weights in enumerate(weights_list):
    # Each weights variable contains the weight matrix and bias vector
    weight_matrix = weights[0]  # The weights
    bias_vector = weights[1]    # The biases
    
    # Plot histogram of weights
    plt.subplot(len(weights_list), 2, i*2+1)
    plt.hist(weight_matrix.flatten(), bins=50, alpha=0.7)
    plt.title(f"Layer {i+1} Weight Distribution")
    plt.xlabel("Weight Value")
    plt.ylabel("Frequency")
    
    # Plot histogram of biases
    plt.subplot(len(weights_list), 2, i*2+2)
    plt.hist(bias_vector, bins=20, alpha=0.7, color='orange')
    plt.title(f"Layer {i+1} Bias Distribution")
    plt.xlabel("Bias Value")
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize weight matrices as heatmaps
# Since the matrices can be large, we'll visualize a sample or the first layer only
def visualize_weight_matrix(weights, title, max_display=64):
    # If the matrix is too large, sample it
    if weights.shape[0] > max_display or weights.shape[1] > max_display:
        # Sample rows and columns
        rows = min(weights.shape[0], max_display)
        cols = min(weights.shape[1], max_display)
        
        # Sample at regular intervals
        row_indices = np.linspace(0, weights.shape[0]-1, rows, dtype=int)
        col_indices = np.linspace(0, weights.shape[1]-1, cols, dtype=int)
        
        # Get the sampled matrix
        sampled_weights = weights[np.ix_(row_indices, col_indices)]
        plt.figure(figsize=(10, 8))
        sns.heatmap(sampled_weights, cmap='viridis', cbar=True)
        plt.title(f"{title} (Sampled {rows}x{cols} from {weights.shape[0]}x{weights.shape[1]})")
    else:
        plt.figure(figsize=(10, 8))
        sns.heatmap(weights, cmap='viridis', cbar=True)
        plt.title(title)
    
    plt.tight_layout()
    plt.show()

# Visualize the first layer weights
first_layer_weights = weights_list[0][0]  # Get the weight matrix from the first layer
visualize_weight_matrix(first_layer_weights, "First Layer Weight Matrix")

### Visualizing Activations

To visualize activations, we need to:
1. Create an input (we'll use a sample from our dataset)
2. Create a model that outputs the activations of a specific layer
3. Get the activations for the input
4. Visualize them

In [ ]:
# Get a sample image
sample_image = x_train[0]
sample_label = y_train[0]

# Display the sample
plt.figure(figsize=(3, 3))
plt.imshow(sample_image, cmap='gray')
plt.title(f"Sample Digit: {sample_label}")
plt.axis('off')
plt.show()

# Create models that output the activations from each layer of the simple model
activation_models = []
for i in range(1, len(simple_model.layers)):
    activation_model = Model(inputs=simple_model.input, 
                            outputs=simple_model.layers[i-1].output)
    activation_models.append(activation_model)

# Prepare the input
sample_flat = sample_image.reshape(1, 784)

# Get activations for each layer
activations = [model.predict(sample_flat) for model in activation_models]

# Plot the activations for each layer
for i, activation in enumerate(activations):
    # Reshape the activation for visualization if needed
    act_reshaped = activation[0]  # Get the first (and only) sample
    
    plt.figure(figsize=(10, 6))
    
    # If this is a large layer, show a sample or a histogram
    if len(act_reshaped) > 100:
        plt.subplot(1, 2, 1)
        plt.plot(act_reshaped[:100])  # Show first 100 activations
        plt.title(f"Layer {i+1} First 100 Activations")
        plt.xlabel("Neuron Index")
        plt.ylabel("Activation")
        
        plt.subplot(1, 2, 2)
        plt.hist(act_reshaped, bins=50, alpha=0.7)
        plt.title(f"Layer {i+1} Activation Distribution")
        plt.xlabel("Activation Value")
        plt.ylabel("Frequency")
    else:
        # For smaller layers, show all activations
        plt.bar(range(len(act_reshaped)), act_reshaped)
        plt.title(f"Layer {i+1} Activations for Digit {sample_label}")
        plt.xlabel("Neuron Index")
        plt.ylabel("Activation Value")
    
    plt.tight_layout()
    plt.show()

## 6. Feature Visualization Techniques

Feature visualization helps us understand what features or patterns the neural network is detecting. Techniques include:
1. **Activation maximization**: Generate inputs that maximize activations of specific neurons
2. **Saliency maps**: Highlight regions of input that influence the output
3. **Class visualization**: Generate inputs that maximize class probabilities

Let's implement some of these techniques:

In [ ]:
# Let's implement a simple saliency map visualization
# Since we haven't trained our model, the results will not be meaningful
# but the technique is still valid

# Train the model with a small subset of data to make the visualization more meaningful
# We'll use only 1000 samples to keep it quick
x_train_small = x_train_flat[:1000]
y_train_small = y_train[:1000]

# Train for just a few epochs
simple_model.fit(
    x_train_small, 
    y_train_small, 
    epochs=2, 
    batch_size=128, 
    verbose=1
)

# Create a saliency map for a given input
def compute_saliency_map(model, input_image, class_idx):
    # Create a TF tensor for the input image
    input_tensor = tf.convert_to_tensor(input_image[np.newaxis, ...], dtype=tf.float32)
    
    # Watch the input tensor
    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        predictions = model(input_tensor)
        loss = predictions[:, class_idx]
    
    # Get the gradients of the loss with respect to the input
    gradients = tape.gradient(loss, input_tensor)
    
    # Take the absolute value to get importance regardless of direction
    gradients = tf.abs(gradients)
    
    # Reduce along color channels
    saliency_map = tf.reduce_max(gradients, axis=-1)
    
    return saliency_map.numpy()[0]

# Select an image for visualization
test_idx = 12
test_image = x_test[test_idx]
test_image_flat = x_test_flat[test_idx:test_idx+1]
true_label = y_test[test_idx]

# Predict the class
pred = simple_model.predict(test_image_flat)
pred_label = np.argmax(pred[0])

# Compute the saliency map
saliency_map = compute_saliency_map(simple_model, test_image_flat, true_label)
saliency_map = saliency_map.reshape(28, 28)

# Plot the original image and the saliency map
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(test_image, cmap='gray')
plt.title(f"Original Image (True: {true_label}, Pred: {pred_label})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(saliency_map, cmap='hot')
plt.title("Saliency Map")
plt.axis('off')

plt.tight_layout()
plt.show()

## 7. Visualizing Model Decision Boundaries

For classification tasks, visualizing decision boundaries helps understand how the model separates different classes. While full MNIST data is 784-dimensional and can't be directly visualized, we can:
1. Use dimensionality reduction techniques like PCA or t-SNE to project to 2D
2. Visualize the decision boundaries in this reduced space

Let's see how to do this:

In [ ]:
# For demonstration purposes, let's use a simpler dataset (digits)
# and reduce it to 2D using t-SNE

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

# Load the digits dataset (smaller than MNIST, with 8x8 images)
digits = load_digits()
X = digits.data
y = digits.target

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create a simple model for this dataset
simple_digits_model = keras.Sequential([
    keras.layers.Input(shape=(64,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

simple_digits_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model briefly
simple_digits_model.fit(X_scaled, y, epochs=5, batch_size=32, validation_split=0.2)

# Use t-SNE to reduce to 2D for visualization
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

# Plot the t-SNE projection colored by the true class
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.title('t-SNE Projection of Digits Dataset')
plt.xlabel('t-SNE Feature 1')
plt.ylabel('t-SNE Feature 2')
plt.show()

In [ ]:
# Now let's attempt to visualize decision boundaries in this 2D t-SNE space
# Note: This is an approximation, as t-SNE is non-linear and the model was trained in the original space

# Train a new model on the t-SNE features
tsne_model = keras.Sequential([
    keras.layers.Input(shape=(2,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

tsne_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
tsne_model.fit(X_tsne, y, epochs=10, batch_size=32, validation_split=0.2)

# Create a mesh grid to visualize the decision boundaries
def plot_decision_boundary(model, X, y):
    h = 0.02  # Step size in the mesh
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predict for each point in the mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = np.argmax(Z, axis=1)
    Z = Z.reshape(xx.shape)
    
    # Plot the decision boundary
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='tab10')
    
    # Plot the data points
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='tab10')
    plt.colorbar(scatter, label='Digit')
    
    plt.title('Decision Boundaries in t-SNE Space')
    plt.xlabel('t-SNE Feature 1')
    plt.ylabel('t-SNE Feature 2')
    plt.tight_layout()
    plt.show()

# Plot the decision boundaries
plot_decision_boundary(tsne_model, X_tsne, y)

## 8. Attention Visualization

For models with attention mechanisms, like many modern deep learning architectures, visualizing attention weights can provide insights into how the model processes information. Though our simple model doesn't have attention mechanisms, let's demonstrate how attention visualizations would work:

In [ ]:
# Simulating attention weights for a sequence model
# This is a simulation since our simple models don't have attention

def visualize_attention(input_text, attention_weights):
    """
    Visualize attention weights for text.
    
    Parameters:
    -----------
    input_text : list of str
        The input text tokens
    attention_weights : numpy.ndarray
        Matrix of attention weights (n_tokens x n_tokens)
    """
    plt.figure(figsize=(10, 8))
    plt.imshow(attention_weights, cmap='Blues')
    
    # Set ticks and labels
    plt.xticks(range(len(input_text)), input_text, rotation=90)
    plt.yticks(range(len(input_text)), input_text)
    
    # Add a colorbar and labels
    plt.colorbar(label='Attention Weight')
    plt.xlabel('Input Token')
    plt.ylabel('Output Token')
    plt.title('Attention Visualization')
    
    # Add text annotations
    for i in range(len(input_text)):
        for j in range(len(input_text)):
            plt.text(j, i, f'{attention_weights[i, j]:.2f}',
                     ha='center', va='center', color='red')
    
    plt.tight_layout()
    plt.show()

# Create sample attention data
sample_text = ["I", "love", "neural", "networks", "visualization"]
n_tokens = len(sample_text)
sample_attention = np.random.rand(n_tokens, n_tokens)
# Make it a proper probability distribution over the input tokens
sample_attention = sample_attention / sample_attention.sum(axis=1, keepdims=True)

# Visualize the attention weights
visualize_attention(sample_text, sample_attention)

## 9. Interpretability Tools and Techniques

Modern interpretability tools help explain neural network predictions. Here we'll demonstrate:
1. **LIME (Local Interpretable Model-agnostic Explanations)**: Explains individual predictions
2. **SHAP (SHapley Additive exPlanations)**: Attributes feature importance
3. **Grad-CAM**: Highlights important regions in images

These require additional libraries, so we'll simulate their outputs for demonstration.

In [ ]:
# Simulate a LIME explanation for an image classification
def simulate_lime_explanation(image, prediction, top_features=5):
    """Simulate LIME explanation highlighting important pixels."""
    # Create a random importance map (in real LIME, this would be based on model output)
    importance = np.random.rand(*image.shape)
    
    # Create a mask of top important regions
    flat_importance = importance.flatten()
    threshold = np.sort(flat_importance)[-int(top_features * flat_importance.size / 100)]
    mask = importance >= threshold
    
    # Create the visualization
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(image, cmap='gray')
    plt.title("Original Image")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(importance, cmap='hot')
    plt.title("Feature Importance")
    plt.axis('off')
    plt.colorbar()
    
    plt.subplot(1, 3, 3)
    # Create a perturbed version showing only important pixels
    highlighted = np.copy(image)
    highlighted[~mask] = 0
    plt.imshow(highlighted, cmap='gray')
    plt.title("Top Important Regions")
    plt.axis('off')
    
    plt.tight_layout()
    plt.suptitle(f"LIME Explanation for Prediction: {prediction}", y=1.05)
    plt.show()

# Simulate LIME explanation for our sample image
simulate_lime_explanation(x_test[test_idx], pred_label)

In [ ]:
# Simulate SHAP values for a neural network prediction
def simulate_shap_values(features, prediction, feature_names=None):
    """Simulate SHAP values for features."""
    if feature_names is None:
        feature_names = [f"Feature {i+1}" for i in range(len(features))]
    
    # Generate random SHAP values (in real usage, these would be calculated)
    shap_values = np.random.randn(len(features))
    
    # Sort by absolute values for visualization
    indices = np.argsort(np.abs(shap_values))[-10:]  # Top 10 features
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), shap_values[indices])
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel('SHAP Value (Impact on Model Output)')
    plt.title(f'SHAP Values for Prediction: {prediction}')
    plt.axvline(x=0, color='gray', linestyle='--')
    plt.tight_layout()
    plt.show()

# Generate some pixel feature names
pixel_features = [f"Pixel ({i//28}, {i%28})" for i in range(784)]

# Simulate SHAP values for the flattened image
simulate_shap_values(x_test_flat[0], pred_label, pixel_features)

## 10. Interactive Visualizations

Interactive visualizations enable dynamic exploration of neural networks. While static plots are useful, interactive ones allow for:
1. **Real-time model inspection**: Adjust parameters and see changes
2. **Deeper exploration**: Drill down into specific model components
3. **Dynamic feature analysis**: See how changing inputs affect outputs

Let's create an interactive visualization using Plotly:

In [ ]:
# Create an interactive t-SNE visualization with Plotly
def interactive_tsne_plot(X_tsne, y, labels=None):
    """Create an interactive t-SNE plot with hover information."""
    if labels is None:
        labels = [f"Digit {i}" for i in range(10)]
    
    # Create a dataframe for Plotly
    df = pd.DataFrame({
        'x': X_tsne[:, 0],
        'y': X_tsne[:, 1],
        'label': [str(labels[i]) for i in y]
    })
    
    # Create the scatter plot
    fig = px.scatter(
        df, x='x', y='y', color='label',
        title='Interactive t-SNE Visualization of Digits',
        labels={'x': 't-SNE Feature 1', 'y': 't-SNE Feature 2', 'label': 'Digit'},
        hover_data={'x': False, 'y': False, 'label': True},
        color_discrete_sequence=px.colors.qualitative.G10
    )
    
    # Update layout
    fig.update_layout(
        legend_title_text='Digit Class',
        height=600
    )
    
    return fig

# Create the interactive plot
interactive_fig = interactive_tsne_plot(X_tsne, y)

# Display the interactive plot
interactive_fig.show()

In [ ]:
# Create a visualization of training metrics over time
def plot_training_history_interactive(history):
    """
    Create an interactive plot of training metrics over epochs.
    """
    # Create simulated training history
    epochs = list(range(1, 11))
    accuracy = [0.2, 0.4, 0.55, 0.65, 0.7, 0.75, 0.78, 0.8, 0.82, 0.83]
    val_accuracy = [0.18, 0.35, 0.5, 0.6, 0.65, 0.68, 0.7, 0.72, 0.73, 0.73]
    loss = [2.0, 1.5, 1.0, 0.8, 0.6, 0.5, 0.4, 0.35, 0.3, 0.28]
    val_loss = [2.1, 1.6, 1.1, 0.9, 0.7, 0.65, 0.6, 0.55, 0.52, 0.51]
    
    # Create the figure with two y-axes
    fig = go.Figure()
    
    # Add traces for accuracy
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=accuracy,
            name="Training Accuracy",
            line=dict(color='blue', width=2),
            mode='lines+markers'
        )
    )
    
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=val_accuracy,
            name="Validation Accuracy",
            line=dict(color='blue', width=2, dash='dash'),
            mode='lines+markers'
        )
    )
    
    # Add traces for loss on secondary y-axis
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=loss,
            name="Training Loss",
            line=dict(color='red', width=2),
            mode='lines+markers',
            yaxis="y2"
        )
    )
    
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=val_loss,
            name="Validation Loss",
            line=dict(color='red', width=2, dash='dash'),
            mode='lines+markers',
            yaxis="y2"
        )
    )
    
    # Create layout with two y-axes
    fig.update_layout(
        title="Training Metrics Over Epochs",
        xaxis=dict(title="Epoch"),
        yaxis=dict(
            title="Accuracy",
            titlefont=dict(color="blue"),
            tickfont=dict(color="blue"),
            range=[0, 1]
        ),
        yaxis2=dict(
            title="Loss",
            titlefont=dict(color="red"),
            tickfont=dict(color="red"),
            anchor="x",
            overlaying="y",
            side="right",
            range=[0, 2.5]
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return fig

# Create the interactive training history plot
history_fig = plot_training_history_interactive(None)  # Using simulated data

# Display the figure
history_fig.show()

## Conclusion

In this notebook, we've explored various techniques for visualizing neural networks:

1. **Network Architecture Visualization**: Understanding the structure of neural networks
2. **Weight and Activation Visualization**: Exploring the internal parameters and neuron activations
3. **Feature Visualization**: Understanding what features the network detects
4. **Decision Boundary Visualization**: Seeing how the model classifies data
5. **Attention Visualization**: Examining attention mechanisms
6. **Interpretability Tools**: Using LIME, SHAP, and other techniques to explain predictions
7. **Interactive Visualizations**: Creating dynamic, exploratory visualizations

These visualization techniques are essential for:
- Understanding how neural networks function
- Debugging model behavior
- Improving model performance
- Explaining models to stakeholders
- Ensuring ethical AI implementation

While we've covered many techniques, there are many more specialized visualizations for specific neural network architectures and use cases. The field of neural network visualization continues to evolve rapidly as models become more complex and the need for interpretability grows.

## References and Further Resources

- Olah, C. et al. "Feature Visualization" *Distill* (2017)
- Zeiler, M. D., & Fergus, R. "Visualizing and Understanding Convolutional Networks" (2014)
- Lundberg, S. M., & Lee, S. I. "A Unified Approach to Interpreting Model Predictions" (2017)
- Ribeiro, M. T., Singh, S., & Guestrin, C. "Why Should I Trust You?: Explaining the Predictions of Any Classifier" (2016)
- Selvaraju, R. R. et al. "Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization" (2017)

**Tools and Libraries:**
- TensorBoard: https://www.tensorflow.org/tensorboard
- Plotly: https://plotly.com/python/
- SHAP: https://github.com/slundberg/shap
- LIME: https://github.com/marcotcr/lime
- Keras Visualization Toolkit: https://github.com/raghakot/keras-vis